# SwinUNETR Segmentation Fine-Tuning

Fine-tune MONAI's `SwinUNETR` on LoG-based pseudo-labels using a
Swin encoder pretrained via SimMIM.

**Notebook structure:**
1. Configuration
2. Imports & device
3. Data loading (images + pseudo-labels)
4. Model (SwinUNETR + pretrained encoder)
5. Training loop
6. Evaluation & saving


## 1. Configuration

All parameters live here. Change these cells, nothing else.

In [ ]:
from types import SimpleNamespace


In [ ]:
cfg = SimpleNamespace(
    seed=42,
    output_root="./outputs",
)


In [ ]:
data_cfg = SimpleNamespace(
    image_root="../../data/patches_128",       # <-- directory with .npy image patches
    label_root="../../data/pseudolabels_128",   # <-- directory with .npy pseudo-label masks
    exclude_patterns=["KONTROLA"],
    val_split=0.1,
    batch_size=16,
    num_workers=2,
    pin_memory=True,
)


In [ ]:
model_cfg = SimpleNamespace(
    in_channels=3,
    out_channels=1,              # binary segmentation (puncta vs background)
    spatial_dims=2,
    img_size=128,
    feature_size=48,             # must match pretrained encoder
    # Path to the pretrained SimMIM encoder weights.
    # This is the file saved by the pretraining notebook (pretrained_encoder_simmim.pt).
    pretrained_encoder_path="./outputs/<your_pretrain_run>/pretrained_encoder_simmim.pt",  # <-- SET THIS
)


In [ ]:
train_cfg = SimpleNamespace(
    epochs=100,
    lr=1e-4,
    weight_decay=0.01,
    warmup_epochs=5,
    grad_clip_norm=1.0,
    experiment_name="finetune_swinunetr_seg",
    model_save_name="best_segmentation_model.pt",
)


## 2. Imports & Device

In [ ]:
import os, sys, time, math, csv, json
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm.auto import tqdm

from monai.networks.nets import SwinUNETR
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric

sys.path.insert(0, os.path.abspath("../.."))


In [ ]:
# Prevent OpenBLAS from spawning too many threads (causes hangs on HPC)
n_cpus = int(os.environ.get("SLURM_CPUS_PER_TASK",
             os.environ.get("PBS_NUM_PPN", 4)))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(max(1, n_cpus // 2)))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(cfg.seed)
generator = torch.Generator().manual_seed(cfg.seed)

if torch.cuda.is_available():
    print(f"{torch.cuda.get_device_name(0)}, "
          f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU")


## 3. Data Loading

**TODO:** Implement a dataset that returns `(image, label)` pairs.
- `image`: `(C, H, W)` float32 in `[0, 1]` (same as pretraining patches)
- `label`: `(1, H, W)` float32 binary mask from LoG pseudo-labeling

The dataset class below is a placeholder. Replace the `__getitem__` body
once pseudo-labels are generated.

In [ ]:
class SegmentationPatchDataset(Dataset):
    """Paired image + pseudo-label dataset.

    TODO: implement loading logic once pseudo-labels are generated.
    Each sample should return:
        image: (C, H, W) float32 in [0, 1]
        label: (1, H, W) float32 binary mask (1 = puncta, 0 = background)
    """

    def __init__(self, image_root, label_root, exclude_patterns=None):
        self.image_root = image_root
        self.label_root = label_root
        # TODO: build list of (image_path, label_path) pairs
        # e.g. by reading index.csv from image_root and finding matching labels
        self.pairs = []  # list of (image_path, label_path)
        raise NotImplementedError(
            "Implement dataset loading once pseudo-labels are available"
        )

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, lbl_path = self.pairs[idx]
        image = torch.from_numpy(np.load(img_path))   # (C, H, W)
        label = torch.from_numpy(np.load(lbl_path))   # (1, H, W) or (H, W)
        if label.ndim == 2:
            label = label.unsqueeze(0)                 # ensure (1, H, W)
        return image, label.float()


In [ ]:
# TODO: uncomment once SegmentationPatchDataset is implemented
#
# dataset = SegmentationPatchDataset(
#     image_root=data_cfg.image_root,
#     label_root=data_cfg.label_root,
#     exclude_patterns=data_cfg.exclude_patterns,
# )
# print(f"Total paired patches: {len(dataset)}")
#
# img, lbl = dataset[0]
# print(f"Image: {img.shape}, dtype={img.dtype}, range=[{img.min():.3f}, {img.max():.3f}]")
# print(f"Label: {lbl.shape}, dtype={lbl.dtype}, unique={lbl.unique().tolist()}")
# assert img.shape[1:] == lbl.shape[1:], "Image and label spatial dims must match"


In [ ]:
# TODO: uncomment once dataset is available
#
# n_val = int(len(dataset) * data_cfg.val_split)
# n_train = len(dataset) - n_val
# train_dataset, val_dataset = random_split(
#     dataset, [n_train, n_val], generator=generator
# )
# print(f"Train: {n_train}, Validation: {n_val}")
#
# train_loader = DataLoader(
#     train_dataset,
#     batch_size=data_cfg.batch_size,
#     shuffle=True,
#     num_workers=data_cfg.num_workers,
#     pin_memory=data_cfg.pin_memory,
#     persistent_workers=data_cfg.num_workers > 0,
#     prefetch_factor=2 if data_cfg.num_workers > 0 else None,
# )
# val_loader = DataLoader(
#     val_dataset,
#     batch_size=data_cfg.batch_size,
#     shuffle=False,
#     num_workers=data_cfg.num_workers,
#     pin_memory=data_cfg.pin_memory,
#     persistent_workers=data_cfg.num_workers > 0,
#     prefetch_factor=2 if data_cfg.num_workers > 0 else None,
# )
# print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")


## 4. Model: SwinUNETR with pretrained encoder

MONAI's `SwinUNETR` is a U-Net with a Swin Transformer encoder and
a CNN decoder with skip connections. We load the SimMIM-pretrained
encoder weights into `model.swinViT`.

Reference: Hatamizadeh et al., "Swin UNETR: Swin Transformers for
Semantic Segmentation of Brain Tumors in MRI Images", BrainLes 2021,
arXiv:2201.01266.
Official: https://github.com/Project-MONAI/research-contributions/tree/main/SwinUNETR

In [ ]:
# SwinUNETR for 2D segmentation.
# img_size, feature_size, spatial_dims must match the pretraining config.
model = SwinUNETR(
    img_size=(model_cfg.img_size, model_cfg.img_size),
    in_channels=model_cfg.in_channels,
    out_channels=model_cfg.out_channels,
    feature_size=model_cfg.feature_size,
    spatial_dims=model_cfg.spatial_dims,
    use_v2=False,  # v1 matches the SwinTransformer used in pretraining
).to(device)

print(f"SwinUNETR created (out_channels={model_cfg.out_channels})")
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")
print(f"Encoder params: {sum(p.numel() for p in model.swinViT.parameters()):,}")


In [ ]:
# Load pretrained SimMIM encoder weights into the SwinViT backbone.
# strict=True because the encoder architecture is identical.
encoder_state = torch.load(model_cfg.pretrained_encoder_path, map_location=device)
msg = model.swinViT.load_state_dict(encoder_state, strict=True)
print(f"Loaded pretrained encoder from: {model_cfg.pretrained_encoder_path}")
print(f"  Missing keys:    {msg.missing_keys}")
print(f"  Unexpected keys: {msg.unexpected_keys}")


## 5. Loss & Metrics

- **DiceCELoss** (MONAI): combines Dice loss + cross-entropy.
  Dice handles class imbalance (puncta are sparse); CE provides
  stable per-pixel gradients early in training.
  Ref: Isensee et al., "nnU-Net" (2021) recommends this combination.
- **DiceMetric**: evaluation metric (not used for backprop).

In [ ]:
# sigmoid=True because out_channels=1 (binary segmentation, logits output).
# If you switch to out_channels=2 (softmax), set sigmoid=False, softmax=True.
loss_fn = DiceCELoss(
    sigmoid=True,
    squared_pred=True,    # use squared denominator in Dice (smoother gradients)
    reduction="mean",
).to(device)

dice_metric = DiceMetric(
    include_background=False,  # only measure foreground (puncta) Dice
    reduction="mean",
)
print(f"Loss: {type(loss_fn).__name__}")


## 6. Training Setup

In [ ]:
def make_lr_lambda(warmup_epochs, total_epochs):
    """Linear warmup then cosine decay to 0."""
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return epoch / max(1, warmup_epochs)
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return lr_lambda


In [ ]:
# Timestamped output directory
run_timestamp = datetime.now().strftime("%Y_%m_%d_%H%M%S")
save_dir = os.path.join(
    cfg.output_root,
    f"{train_cfg.experiment_name}_{run_timestamp}"
)
os.makedirs(save_dir, exist_ok=True)
print(f"Output directory: {save_dir}")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=train_cfg.lr,
    weight_decay=train_cfg.weight_decay,
    betas=(0.9, 0.999),
)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    make_lr_lambda(train_cfg.warmup_epochs, train_cfg.epochs),
)
scaler = torch.amp.GradScaler(device.type, enabled=device.type == "cuda")


### Experiment metadata + CSV logger

In [ ]:
experiment_meta = {
    "experiment_name":        train_cfg.experiment_name,
    "start_time":             datetime.now().isoformat(),
    "device":                 str(device),
    "gpu_name":               torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "pytorch_version":        torch.__version__,

    "n_train":                len(train_loader.dataset),
    "n_val":                  len(val_loader.dataset),
    "img_size":               model_cfg.img_size,
    "in_channels":            model_cfg.in_channels,
    "out_channels":           model_cfg.out_channels,
    "pretrained_encoder":     model_cfg.pretrained_encoder_path,

    "total_params":           sum(p.numel() for p in model.parameters()),
    "encoder_params":         sum(p.numel() for p in model.swinViT.parameters()),

    "loss_fn":                "DiceCELoss(sigmoid=True)",
    "optimizer":              "AdamW",
    "lr":                     train_cfg.lr,
    "weight_decay":           train_cfg.weight_decay,
    "batch_size":             data_cfg.batch_size,
    "epochs":                 train_cfg.epochs,
    "warmup_epochs":          train_cfg.warmup_epochs,
    "scheduler":              "linear warmup + cosine decay",
    "mixed_precision":        device.type == "cuda",
    "grad_clip_norm":         train_cfg.grad_clip_norm,
    "seed":                   cfg.seed,
}

print("=" * 60)
for k, v in experiment_meta.items():
    print(f"  {k:30s}: {v}")
print("=" * 60)

csv_path = os.path.join(save_dir, f"{train_cfg.experiment_name}_log.csv")
csv_fields = [
    "epoch", "train_loss", "val_loss", "val_dice", "lr",
    "epoch_time_s", "train_time_s", "val_time_s",
    "best_val_dice", "best_epoch",
    "grad_norm_mean", "grad_norm_max",
]
csv_file = open(csv_path, "w", newline="")
csv_writer = csv.DictWriter(csv_file, fieldnames=csv_fields)
csv_writer.writeheader()
print(f"Logging to: {csv_path}")


### (Optional) Resume from checkpoint

Skip this cell for a fresh run. Set `resume_path` to continue from a saved checkpoint.

In [ ]:
resume_path = None  # e.g. "./outputs/finetune_.../best_segmentation_model.pt"

start_epoch = 1

if resume_path is not None:
    ckpt = torch.load(resume_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    scaler.load_state_dict(ckpt["scaler_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    print(f"Resumed from epoch {ckpt['epoch']}, "
          f"val_dice={ckpt.get('val_dice', 'N/A')}")
else:
    print("Fresh training run")


### Training loop

In [ ]:
best_val_dice  = 0.0
best_epoch     = 0
train_losses   = []
val_losses     = []
val_dices      = []
lr_history     = []
grad_norms     = []
epoch_times    = []
total_train_start = time.time()


In [ ]:
for epoch in range(start_epoch, train_cfg.epochs + 1):
    # --- Training ---
    model.train()
    running_loss = 0.0
    epoch_grad_norms = []
    train_start = time.time()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{train_cfg.epochs} [train]", leave=False)
    for batch_idx, (images, labels) in enumerate(pbar, 1):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device.type, enabled=device.type == "cuda"):
            logits = model(images)           # (B, out_channels, H, W)
            loss = loss_fn(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        total_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=train_cfg.grad_clip_norm
        )
        epoch_grad_norms.append(total_norm.item())
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        pbar.set_postfix(loss=f"{running_loss / batch_idx:.6f}")

    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)
    train_time = time.time() - train_start
    scheduler.step()

    # --- Validation ---
    model.eval()
    val_running_loss = 0.0
    dice_metric.reset()
    val_start = time.time()

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch}/{train_cfg.epochs} [val]", leave=False):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast(device.type, enabled=device.type == "cuda"):
                logits = model(images)
                loss = loss_fn(logits, labels)
            val_running_loss += loss.item()

            # Dice metric expects discrete predictions
            preds = (torch.sigmoid(logits) > 0.5).float()
            dice_metric(y_pred=preds, y=labels)

    val_loss = val_running_loss / len(val_loader)
    val_losses.append(val_loss)
    val_dice = dice_metric.aggregate().item()
    val_dices.append(val_dice)
    val_time = time.time() - val_start

    # --- Logging ---
    current_lr = scheduler.get_last_lr()[0]
    lr_history.append(current_lr)
    epoch_time = train_time + val_time
    epoch_times.append(epoch_time)
    mean_grad = sum(epoch_grad_norms) / len(epoch_grad_norms)
    max_grad  = max(epoch_grad_norms)
    grad_norms.append(mean_grad)

    improved = ""
    if val_dice > best_val_dice:
        best_val_dice = val_dice
        best_epoch = epoch
        improved = " *best*"
        torch.save({
            "epoch":               epoch,
            "model_state_dict":    model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict":   scaler.state_dict(),
            "val_loss":            val_loss,
            "val_dice":            val_dice,
            "train_loss":          train_loss,
            "model_cfg":           vars(model_cfg),
            "train_cfg":           {k: v for k, v in vars(train_cfg).items()
                                    if isinstance(v, (int, float, str, bool))},
        }, os.path.join(save_dir, train_cfg.model_save_name))

    print(f"Epoch {epoch:3d}/{train_cfg.epochs} | "
          f"Train: {train_loss:.6f} | Val: {val_loss:.6f} | "
          f"Dice: {val_dice:.4f} | "
          f"LR: {current_lr:.2e} | "
          f"Time: {epoch_time:.1f}s{improved}")

    csv_writer.writerow({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_dice": val_dice,
        "lr": current_lr,
        "epoch_time_s": epoch_time,
        "train_time_s": train_time,
        "val_time_s": val_time,
        "best_val_dice": best_val_dice,
        "best_epoch": best_epoch,
        "grad_norm_mean": mean_grad,
        "grad_norm_max": max_grad,
    })
    csv_file.flush()

total_train_time = time.time() - total_train_start
csv_file.close()
print(f"\nTotal training time: {total_train_time / 60:.1f} min")


### Save experiment summary

In [ ]:
experiment_meta["end_time"]          = datetime.now().isoformat()
experiment_meta["total_time_hours"]  = round(total_train_time / 3600, 3)
experiment_meta["best_val_dice"]     = best_val_dice
experiment_meta["best_epoch"]        = best_epoch
experiment_meta["final_train_loss"]  = train_losses[-1]
experiment_meta["final_val_loss"]    = val_losses[-1]
experiment_meta["final_val_dice"]    = val_dices[-1]

meta_path = os.path.join(save_dir, f"{train_cfg.experiment_name}_meta.json")
with open(meta_path, "w") as f:
    json.dump(experiment_meta, f, indent=2)
print(f"Saved metadata: {meta_path}")


### Loss & Dice curves

In [ ]:
epochs_range = range(1, len(train_losses) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(epochs_range, train_losses, label="Train")
axes[0].plot(epochs_range, val_losses,   label="Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("DiceCE Loss")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, val_dices, label="Val Dice", color="green")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Dice")
axes[1].set_title("Validation Dice")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_range, lr_history, color="orange")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("LR")
axes[2].set_title("Learning Rate")
axes[2].grid(True, alpha=0.3)

plt.suptitle(f"Segmentation: {train_cfg.experiment_name}")
plt.tight_layout()
fig.savefig(os.path.join(save_dir, "training_curves.pdf"), bbox_inches="tight")
fig.savefig(os.path.join(save_dir, "training_curves.png"), dpi=200, bbox_inches="tight")
plt.show()
